In [0]:
import os
import requests
import json
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql.functions import col

In [0]:
def get_post_media_item_df(spark):
    df = spark.sql("""
                    select distinct post_id, media_id
                    from bz_cn_dl.datalake_community.dw_cust_community_tb_post_media
                    where delete_at is null
                  """
    ).toPandas()
    return df

def get_media_item_ids_by_chunk(df, start_rn, end_rn):
    media_ids = df.iloc[start_rn:end_rn]['media_id'].tolist()
    return media_ids

def convert_long_to_int(sdf, columns):
    # convert LongType column to IntegerType
    for column in columns:
      sdf = sdf.withColumn(column, col(column).cast("int"))
    return sdf

def upsert_to_delta_table(spark_df, target_table_name, merge_keys):
    """
    Simple upsert function to merge a Spark DataFrame into a Delta table
    
    Args:
        spark_df: Source Spark DataFrame
        target_table_name: the target Delta table
        merge_keys: List of column names to use as merge keys
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Check if the target table is empty 
        if spark.table(target_table_name).isEmpty():
            # write spark df to the table
            spark_df.write \
                .format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(target_table_name)
            return True
        else:
            delta_table = DeltaTable.forName(spark, target_table_name)
            # Build merge condition
            merge_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_keys])
          
            # Perform merge operation
            delta_table.alias("target") \
                .merge(spark_df.alias("source"), merge_condition) \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()
          
            print("Upsert operation completed successfully")
            return True
        
    except Exception as e:
        print(f"Error during upsert operation: {str(e)}")
        return False
      
def post_data_to_media_item_index_table(spark, media_ids, host, api_key, post_media_df, target_table_name):
    headers = {"Content-Type": "application/json", "apikey": api_key}
    data = {'ids':media_ids}
    response = requests.post(host, headers=headers, json=data)

    try:
      # Parse the response as JSON
      response_json = response.json()

      # Extract the value for a given field
      m_ids = [file["id"] for file in response_json["data"]["medias"]]
      urls = [file["url"] for file in response_json["data"]["medias"]]
      file_types = [file["type"] for file in response_json["data"]["medias"]]
      
      df = pd.DataFrame({"media_id":m_ids,
                         "url":urls,
                         "file_type":file_types})
      df = df.merge(post_media_df, on='media_id', how='inner')
      df = df[['post_id','media_id','url','file_type']] 

      sdf = spark.createDataFrame(df)
      sdf = convert_long_to_int(sdf, ["file_type"])
      upsert_to_delta_table(sdf, target_table_name, merge_keys=['post_id','media_id'])

    except requests.RequestException as e:
      raise RuntimeError(f"POST request failed: {e}")

    return None

In [0]:
host = dbutils.secrets.get(scope="dmp_secrets", key="PDLKCOMMUNITYURL")
api_key = dbutils.secrets.get(scope="dmp_secrets", key="PDLKCOMMUNITYURLAPIKEY")

config_path = dbutils.widgets.get("config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']
target_table_name = db+'.'+table_names["media_item_index"]
chunk_size = 10000 # Fetch a chunk of data due to the file size limit of api request 

post_media_df = get_post_media_item_df(spark)
for i in range(0, len(post_media_df), chunk_size):
    media_ids = get_media_item_ids_by_chunk(post_media_df, start_rn=i, end_rn=i+chunk_size)
    post_data_to_media_item_index_table(spark, media_ids, host, api_key, post_media_df, target_table_name)